# RanPAC Phase 2 · Đối chiếu Table A5 và Khảo sát độ nhạy tham số điều chuẩn ($\lambda$)

Notebook này thuộc bản **tái cài đặt từ mô tả bài báo (Re-implementation)** trên **Split CIFAR-100** (10 task × 10 lớp, seed `1993`, ViT-B/16 ImageNet-21K đóng băng, không dùng Phase 1), phục vụ đồng thời hai yêu cầu của đồ án:

1. **Phần A — Experimental Reproduction (Table A5, Appendix F.5):**
   - Trong [05_ranpac_m_sweep.ipynb](05_ranpac_m_sweep.ipynb), nhóm đã sweep $M$ nhưng dùng **CV $\lambda$ theo từng task**. Trong khi đó, **Table A5** của bài báo cố định **$\lambda = 100$** cho mọi $M \in \{100, 200, 400, 800, 1250, 2500, 5000, 10000, 15000\}$.
   - Phần A chạy đúng cấu hình $\lambda = 100$ cố định với tag `reproduction`, xuất bảng đối chiếu 3 cột: **Paper Table A5 ($\lambda=100$)** vs **Re-implement ($\lambda=100$)** vs **Re-implement (Task-CV $\lambda$)**.

2. **Phần B — Critical Analysis & Empirical Demonstration (Sensitivity Sweep theo $\lambda$):**
   - **Điểm yếu lý thuyết & công thức:** Ở task $t$, Algorithm 1 chọn $\lambda_t$ bằng 80/20 split chỉ trên dữ liệu của **riêng task hiện tại** ($4000$ mẫu fit), nơi trị riêng ma trận Gram con cỡ $\Theta(N_{\text{task}})$, nhưng lại áp $\lambda_t$ vào hệ tích lũy $(G_t + \lambda_t I) W_{\text{out}} = C_t$ có chuẩn $G_t$ tăng tuyến tính theo $t \cdot N_{\text{task}}$ (lớn gấp $10\times$ ở task cuối).
   - **Chi phí thực tế:** Ở $M = 10000$, bước tìm $\lambda$ chiếm hơn **60% tổng thời gian fit + eval** (~338s / 543s).
   - Phần B quét $\lambda$ cố định trên lưới rộng $\{10^{-4}, 10^{-2}, 1, 10, 10^2, 10^3, 10^4, 10^5, 10^6, 10^8\}$ tại $M \in \{2500, 10000\}$ với tag `empirics`, đo đường cong suy giảm $A[10]$, quỹ đạo $\lambda_t$ của CV, và đánh đổi **Accuracy vs Thời gian tính toán**.


## 1. Cài đặt môi trường và chuẩn bị Feature Cache dùng chung

Đặt `ranpac_core.py`, `ranpac_data.py`, `ranpac_colab.py`, `ranpac_full.py` và `test_ranpac_core.py` trong `MyDrive/RanPAC_implement`. Giữ cùng `WORK_DIR` với các notebook `01`–`05` để **tái sử dụng trực tiếp cache embedding ViT-B/16** (không tốn thời gian trích xuất lại trên GPU). Mọi run đều log về project W&B công khai `pnp123/RanPAC-reproduction` và gắn tag phân biệt rõ `reproduction` và `empirics`.


In [ ]:
%pip install -q timm==1.0.22 wandb==0.31.0


In [ ]:
import gc
import json
import sys
import unittest
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import torch
from google.colab import drive

drive.mount('/content/drive')
SOURCE_DIR = Path('/content/drive/MyDrive/RanPAC_implement')
WORK_DIR = Path('/content/drive/MyDrive/ranpac-runs')
DATA_ROOT = '/content/ranpac-data'

if not (SOURCE_DIR / 'ranpac_colab.py').is_file():
    raise FileNotFoundError(f'Chưa tìm thấy mã nguồn tại {SOURCE_DIR}')
if str(SOURCE_DIR) not in sys.path:
    sys.path.insert(0, str(SOURCE_DIR))

from ranpac_colab import TABLE_A5_PAPER, configure_wandb, run_phase2
from ranpac_core import NUM_TASKS, SEED, split_class_order
from ranpac_data import FeatureCache, atomic_output, feature_config


In [ ]:
suite = unittest.defaultTestLoader.loadTestsFromName('test_ranpac_core')
test_result = unittest.TextTestRunner(verbosity=2).run(suite)
if not test_result.wasSuccessful():
    raise AssertionError('Kiểm tra unit test trên CPU chưa đạt.')


## 2. Cấu hình chung và kiểm tra cache embedding

- `RUN_TABLE_A5`: bật/tắt Phần A (đối chiếu Table A5 với $\lambda = 100$ cố định). Mặc định chạy 8 mốc từ `100` đến `10000`; bật `INCLUDE_M15000 = True` nếu runtime Colab có đủ RAM hệ thống (>12 GB trống, vì riêng ma trận $G$ FP32 ở $M=15000$ chiếm ~860 MiB).
- `RUN_LAMBDA_SWEEP`: bật/tắt Phần B (quét độ nhạy $\lambda$ cố định tại `M = 2500` và `M = 10000`).
- Pipeline tự động kiểm tra từng run: nếu một mốc đã chạy xong đủ 10 task trong cùng `RUN_LABEL` thì giữ nguyên kết quả và chuyển sang mốc kế tiếp; nếu bị ngắt giữa chừng thì tự động `resume=True`.


In [ ]:
WANDB_ENTITY = 'pnp123'
WANDB_PROJECT = 'RanPAC-reproduction'
WANDB_MODE = 'online'          # 'online' hoặc 'offline'
RUN_LABEL = 'lam-study-01'

BATCH_SIZE = 48
NUM_WORKERS = 2
STATS_DEVICE = 'cpu'
RIDGE_NO_RP_A10_PCT = 87.02    # Mốc Ridge không RP từ Notebook 02

RUN_TABLE_A5 = True
INCLUDE_M15000 = False
A5_DIMS = tuple(m for m in TABLE_A5_PAPER if m < 15000 or INCLUDE_M15000)

RUN_LAMBDA_SWEEP = True
LAMBDA_SWEEP_DIMS = (2500, 10000)
LAMBDA_SWEEP_VALUES = (1e-4, 1e-2, 1.0, 10.0, 100.0, 1e3, 1e4, 1e5, 1e6, 1e8)

# Tuỳ chọn đường dẫn tới checkpoint CV-lambda đã chạy trước đó để khỏi chạy lại đường đối chiếu CV:
EXISTING_CV_RUN_DIRS = {
    2500: WORK_DIR / 'runs' / f'rp-sweep-m2500-s{SEED}-m-sweep-01',
    10000: WORK_DIR / 'runs' / f'rp-table1-s{SEED}-main',
}

if not torch.cuda.is_available():
    raise RuntimeError('Chọn GPU runtime trên Colab trước khi chạy.')
if WANDB_MODE == 'online':
    configure_wandb(WANDB_ENTITY, WANDB_MODE)

feature_device = torch.device('cuda')
cache = FeatureCache(
    WORK_DIR / 'embeddings', DATA_ROOT, feature_config(split_class_order(), feature_device),
    feature_device, BATCH_SIZE, NUM_WORKERS,
)
for task in range(1, NUM_TASKS + 1):
    for split in ('train', 'test'):
        shard = cache.root / f'{split}-task-{task:02}.pt'
        if not shard.is_file():
            print(f'Trích embedding {split} cho task {task:02}...', flush=True)
            cache.load(split, task)
print(f'Feature cache sẵn sàng tại: {cache.root}')
del cache
gc.collect()
torch.cuda.empty_cache()


## 3. Phần A — Chạy và đối chiếu trực tiếp **Table A5** ($\lambda = 100$ cố định)

Mỗi mức $M$ trong `A5_DIMS` dùng preset `rp-a5-m{M}` (`method='ridge-rp'`, `activation='relu'`, `fixed_lambda=100.0`), log vào W&B group `cifar100-phase2-table-A5` với tag `['ridge-rp', 'reproduction', 'table-a5']`. Vì không cần chạy bước tìm $\lambda$ bằng phân rã trị riêng trên 80/20 split, tốc độ chạy nhanh hơn đáng kể so với [05_ranpac_m_sweep.ipynb](05_ranpac_m_sweep.ipynb).


In [ ]:
def ensure_phase2_complete(preset, label, *, fixed_lambda_override=None, extra_tags=None, group=None):
    run_dir = WORK_DIR / 'runs' / f'{preset}-s{SEED}-{label}'
    ckpt_path = run_dir / 'checkpoint.pt'
    if ckpt_path.is_file():
        ckpt = torch.load(ckpt_path, map_location='cpu', weights_only=True)
        if len(ckpt.get('records', ())) == NUM_TASKS:
            print(f'[Đã xong] {run_dir.name} -> A[10] = {ckpt["records"][-1]["avg_accuracy_pct"]:.2f}%')
            return ckpt['records'], run_dir, ckpt['wandb'].get('url')
        resume_flag = True
    else:
        resume_flag = False

    records = run_phase2(
        preset, WORK_DIR, entity=WANDB_ENTITY, project=WANDB_PROJECT,
        stop_after_task=NUM_TASKS, run_label=label, resume=resume_flag,
        wandb_mode=WANDB_MODE, batch_size=BATCH_SIZE, workers=NUM_WORKERS,
        stats_device=STATS_DEVICE, fixed_lambda_override=fixed_lambda_override,
        extra_tags=extra_tags, wandb_group_override=group,
    )
    ckpt = torch.load(ckpt_path, map_location='cpu', weights_only=True)
    return records, run_dir, ckpt['wandb'].get('url')


a5_rows = []
if RUN_TABLE_A5:
    for dim in A5_DIMS:
        preset = f'rp-a5-m{dim}'
        records, run_dir, url = ensure_phase2_complete(
            preset, RUN_LABEL,
            extra_tags=['reproduction', 'table-a5'],
            group=f'cifar100-phase2-table-A5-{RUN_LABEL}',
        )
        paper_acc = TABLE_A5_PAPER[dim]
        reimp_acc = records[-1]['avg_accuracy_pct']
        fit_eval_s = sum(r['fit_seconds'] + r['eval_seconds'] for r in records)
        solve_s = sum(r['ridge_solve_seconds'] or 0.0 for r in records)
        a5_rows.append({
            'M': dim,
            'paper_a5_pct': paper_acc,
            'reimp_fixed100_pct': reimp_acc,
            'delta_vs_paper_pp': reimp_acc - paper_acc,
            'fit_eval_seconds': fit_eval_s,
            'ridge_solve_seconds': solve_s,
            'peak_rss_mib': max((r.get('host_peak_rss_mib') or 0.0) for r in records),
            'wandb_url': url,
        })

    df_a5 = pd.DataFrame(a5_rows)
    cv_csv = SOURCE_DIR / 'sweep_summary.csv'
    if cv_csv.is_file():
        df_cv = pd.read_csv(cv_csv)[['M', 'a10_pct', 'fit_eval_seconds']].rename(columns={
            'a10_pct': 'reimp_task_cv_pct',
            'fit_eval_seconds': 'cv_fit_eval_seconds',
        })
        df_a5 = df_a5.merge(df_cv, on='M', how='left')
        df_a5['cv_minus_fixed100_pp'] = df_a5['reimp_task_cv_pct'] - df_a5['reimp_fixed100_pct']

    summary_dir = WORK_DIR / 'sweeps' / f'lambda-study-s{SEED}-{RUN_LABEL}'
    with atomic_output(summary_dir / 'table_a5_comparison.csv') as tmp:
        df_a5.to_csv(tmp, index=False)
    display(df_a5.round(3))

    fig, axes = plt.subplots(1, 2, figsize=(12, 4.2), layout='constrained')
    axes[0].plot(df_a5['M'], df_a5['paper_a5_pct'], 'k^--', label='Paper Table A5 (λ=100)')
    axes[0].plot(df_a5['M'], df_a5['reimp_fixed100_pct'], 'o-', label='Re-implement (λ=100 cố định)')
    if 'reimp_task_cv_pct' in df_a5:
        sub = df_a5.dropna(subset=['reimp_task_cv_pct'])
        axes[0].plot(sub['M'], sub['reimp_task_cv_pct'], 's-', label='Re-implement (CV λ theo task)')
    axes[0].axhline(RIDGE_NO_RP_A10_PCT, linestyle=':', color='gray', label='Ridge không RP (87.02%)')
    axes[0].set_xscale('log')
    axes[0].set_xlabel('Kích thước projection M')
    axes[0].set_ylabel('A[10] (%)')
    axes[0].set_title('Đối chiếu Table A5: Paper vs Re-implement')
    axes[0].legend()
    axes[0].grid(alpha=0.25)

    axes[1].plot(df_a5['M'], df_a5['fit_eval_seconds'], 'o-', label='Fit + eval (λ=100 cố định)')
    if 'cv_fit_eval_seconds' in df_a5:
        sub = df_a5.dropna(subset=['cv_fit_eval_seconds'])
        axes[1].plot(sub['M'], sub['cv_fit_eval_seconds'], 's--', label='Fit + eval (CV λ theo task)')
    axes[1].set_xscale('log')
    axes[1].set_xlabel('Kích thước projection M')
    axes[1].set_ylabel('Thời gian 10 task (giây)')
    axes[1].set_title('Chi phí thời gian khi bỏ bước tìm λ')
    axes[1].legend()
    axes[1].grid(alpha=0.25)

    with atomic_output(summary_dir / 'table_a5_comparison.png') as tmp:
        fig.savefig(tmp, format='png', dpi=150)
    plt.show()


## 4. Phần B — Sensitivity Sweep theo $\lambda$ cố định vs Task-specific CV $\lambda$ (Empirical Demonstration)

Với mỗi $M \in \{2500, 10000\}$:
- Chạy tất cả các giá trị $\lambda \in \{10^{-4}, 10^{-2}, 1, 10, 10^2, 10^3, 10^4, 10^5, 10^6, 10^8\}$ với `extra_tags=['empirics', 'lambda-sweep']`.
- Đối chiếu với đường **CV $\lambda$ theo task** (tận dụng checkpoint sẵn có ở `EXISTING_CV_RUN_DIRS` nếu tìm thấy trên Drive, hoặc tự chạy mới nếu chưa có) và vẽ quỹ đạo $\lambda_t$ mà thuật toán chọn qua 10 task.


In [ ]:
def lambda_slug(val):
    exp = int(round(torch.log10(torch.tensor(float(val))).item()))
    if abs(10.0 ** exp - val) / max(val, 1e-12) < 1e-6:
        return f'1e{exp:+d}'.replace('+', 'p').replace('-', 'm')
    return str(val).replace('.', 'p')


def load_or_run_cv_baseline(dim):
    candidate = EXISTING_CV_RUN_DIRS.get(dim)
    if candidate is not None and (candidate / 'checkpoint.pt').is_file():
        ckpt = torch.load(candidate / 'checkpoint.pt', map_location='cpu', weights_only=True)
        if len(ckpt.get('records', ())) == NUM_TASKS:
            print(f'[Dùng lại CV baseline] M={dim} từ {candidate.name}')
            return ckpt['records'], ckpt['wandb'].get('url')
    preset = 'rp-table1' if dim == 10000 else f'rp-sweep-m{dim}'
    records, _, url = ensure_phase2_complete(
        preset, f'{RUN_LABEL}-cv-m{dim}',
        extra_tags=['empirics', 'lambda-sweep', 'task-cv'],
        group=f'cifar100-lambda-sweep-{RUN_LABEL}',
    )
    return records, url


if RUN_LAMBDA_SWEEP:
    sweep_rows = []
    cv_trajectories = []
    cv_baselines = {}

    for dim in LAMBDA_SWEEP_DIMS:
        cv_records, cv_url = load_or_run_cv_baseline(dim)
        cv_a10 = cv_records[-1]['avg_accuracy_pct']
        cv_time = sum(r['fit_seconds'] + r['eval_seconds'] for r in cv_records)
        cv_search_time = sum(r['lambda_search_seconds'] or 0.0 for r in cv_records)
        cv_baselines[dim] = {
            'a10_pct': cv_a10,
            'fit_eval_seconds': cv_time,
            'lambda_search_seconds': cv_search_time,
            'wandb_url': cv_url,
        }
        for r in cv_records:
            cv_trajectories.append({
                'M': dim,
                'task': r['task'],
                'selected_lambda': r['ridge_lambda'],
                'avg_accuracy_pct': r['avg_accuracy_pct'],
            })

        base_preset = f'rp-sweep-m{dim}'
        for lam in LAMBDA_SWEEP_VALUES:
            slug = lambda_slug(lam)
            label = f'{RUN_LABEL}-lam-{slug}'
            records, _, url = ensure_phase2_complete(
                base_preset, label,
                fixed_lambda_override=lam,
                extra_tags=['empirics', 'lambda-sweep'],
                group=f'cifar100-lambda-sweep-{RUN_LABEL}',
            )
            a10 = records[-1]['avg_accuracy_pct']
            fit_eval_s = sum(r['fit_seconds'] + r['eval_seconds'] for r in records)
            solve_s = sum(r['ridge_solve_seconds'] or 0.0 for r in records)
            sweep_rows.append({
                'M': dim,
                'lambda_fixed': lam,
                'a10_pct': a10,
                'cv_a10_pct': cv_a10,
                'delta_vs_cv_pp': a10 - cv_a10,
                'fit_eval_seconds': fit_eval_s,
                'cv_fit_eval_seconds': cv_time,
                'speedup_vs_cv_x': cv_time / max(fit_eval_s, 1e-6),
                'ridge_solve_seconds': solve_s,
                'wandb_url': url,
            })

    df_lam = pd.DataFrame(sweep_rows)
    df_traj = pd.DataFrame(cv_trajectories)
    summary_dir = WORK_DIR / 'sweeps' / f'lambda-study-s{SEED}-{RUN_LABEL}'
    with atomic_output(summary_dir / 'lambda_sweep_summary.csv') as tmp:
        df_lam.to_csv(tmp, index=False)
    with atomic_output(summary_dir / 'cv_lambda_trajectory.csv') as tmp:
        df_traj.to_csv(tmp, index=False)

    display(df_lam.round(3))

    fig, axes = plt.subplots(1, 3, figsize=(15, 4.3), layout='constrained')
    for dim, marker in zip(LAMBDA_SWEEP_DIMS, ('o', 's')):
        sub = df_lam[df_lam['M'] == dim].sort_values('lambda_fixed')
        axes[0].plot(sub['lambda_fixed'], sub['a10_pct'], f'{marker}-', label=f'Fixed λ (M={dim})')
        axes[0].axhline(cv_baselines[dim]['a10_pct'], linestyle='--', alpha=0.8,
                        label=f'Task-CV λ (M={dim}: {cv_baselines[dim]["a10_pct"]:.2f}%)')
    axes[0].axhline(RIDGE_NO_RP_A10_PCT, linestyle=':', color='gray', label='Ridge không RP (87.02%)')
    axes[0].set_xscale('log')
    axes[0].set_xlabel('Giá trị λ cố định')
    axes[0].set_ylabel('A[10] (%)')
    axes[0].set_title('Độ nhạy A[10] theo λ cố định')
    axes[0].legend(fontsize=8)
    axes[0].grid(alpha=0.25)

    for dim, marker in zip(LAMBDA_SWEEP_DIMS, ('o', 's')):
        traj = df_traj[df_traj['M'] == dim]
        axes[1].plot(traj['task'], traj['selected_lambda'], f'{marker}-', label=f'CV λ_t (M={dim})')
    axes[1].set_yscale('log')
    axes[1].set_xticks(range(1, NUM_TASKS + 1))
    axes[1].set_xlabel('Task t')
    axes[1].set_ylabel('λ_t được chọn trên 80/20 task t')
    axes[1].set_title('Quỹ đạo λ_t của Task-CV qua 10 task')
    axes[1].legend()
    axes[1].grid(alpha=0.25)

    bar_labels, search_times, other_times = [], [], []
    for dim in LAMBDA_SWEEP_DIMS:
        bar_labels.append(f'M={dim}\nTask-CV')
        s_t = cv_baselines[dim]['lambda_search_seconds']
        tot_t = cv_baselines[dim]['fit_eval_seconds']
        search_times.append(s_t)
        other_times.append(max(tot_t - s_t, 0.0))

        sub100 = df_lam[(df_lam['M'] == dim) & (df_lam['lambda_fixed'] == 100.0)]
        if not sub100.empty:
            bar_labels.append(f'M={dim}\nFixed λ=100')
            search_times.append(0.0)
            other_times.append(float(sub100['fit_eval_seconds'].iloc[0]))
    axes[2].bar(bar_labels, other_times, label='Solve + Eval')
    axes[2].bar(bar_labels, search_times, bottom=other_times, label='Tìm λ (80/20 CV)')
    axes[2].set_ylabel('Thời gian 10 task (giây)')
    axes[2].set_title('Tỷ trọng thời gian tìm λ vs Solve + Eval')
    axes[2].legend()
    axes[2].grid(axis='y', alpha=0.25)

    with atomic_output(summary_dir / 'lambda_sensitivity_analysis.png') as tmp:
        fig.savefig(tmp, format='png', dpi=150)
    plt.show()
    print(f'Đã lưu toàn bộ CSV và biểu đồ tại: {summary_dir}')
